# Telco Customer Churn Prediction

This notebook builds an end-to-end churn analysis workflow using a synthetic Telco-style customer dataset. It covers exploratory analysis, data cleaning, one-hot encoding, Random Forest modeling, recall-focused tuning, ROC-AUC evaluation, feature importance review, and K-Means customer segmentation.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
DATA_PATH = Path("Telco_customer_churn.xlsx")


## 1. Load the Dataset

The workbook contains customer demographics, subscriptions, billing details, churn labels, churn scores, and CLTV values. The target variable is `Churn Value`.


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="CustomerData")
print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df[["Tenure Months", "Monthly Charges", "Total Charges", "Churn Value", "Churn Score", "CLTV"]].describe()


## 2. Exploratory Data Analysis

Start with churn distribution, then compare churn against tenure, charges, contract type, internet service, payment method, and tech support.


In [ ]:
churn_counts = df["Churn Label"].value_counts()
churn_rate = df["Churn Value"].mean()

display(churn_counts.to_frame("Customers"))
print(f"Overall churn rate: {churn_rate:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x="Churn Label", ax=axes[0])
axes[0].set_title("Churn Distribution")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Customers")

sns.histplot(data=df, x="Tenure Months", hue="Churn Label", bins=24, kde=True, ax=axes[1])
axes[1].set_title("Tenure Distribution by Churn")
axes[1].set_xlabel("Tenure Months")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=df, x="Churn Label", y="Tenure Months", ax=axes[0])
axes[0].set_title("Tenure vs Churn")

sns.boxplot(data=df, x="Churn Label", y="Monthly Charges", ax=axes[1])
axes[1].set_title("Monthly Charges vs Churn")

plt.tight_layout()
plt.show()


In [ ]:
categorical_views = ["Contract", "Internet Service", "Payment Method", "Tech Support"]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for column, ax in zip(categorical_views, axes.ravel()):
    order = df[column].value_counts().index
    sns.countplot(data=df, y=column, hue="Churn Label", order=order, ax=ax)
    ax.set_title(f"{column} by Churn")
    ax.set_xlabel("Customers")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "Churn Value", "Churn Score", "CLTV"]

plt.figure(figsize=(8, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="vlag", center=0, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn Label"], normalize="index").sort_values("Yes", ascending=False)
contract_churn.style.format("{:.1%}")


## 3. Data Cleaning

`Total Charges` is converted to numeric, missing values are filled with zero, and model leakage / identifier columns are removed before encoding.


In [ ]:
working_df = df.copy()
working_df["Total Charges"] = pd.to_numeric(working_df["Total Charges"], errors="coerce").fillna(0)

drop_columns = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "City",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "CLTV",
    "Churn Reason",
]

model_df = working_df.drop(columns=[col for col in drop_columns if col in working_df.columns])
model_df.head()


## 4. Encoding and Feature Selection Setup

All categorical fields are one-hot encoded. The target column is excluded from the feature matrix.


In [ ]:
X = model_df.drop(columns=["Churn Value"])
y = model_df["Churn Value"].astype(int)

X_encoded = pd.get_dummies(X, drop_first=True)
print("Encoded feature shape:", X_encoded.shape)
X_encoded.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape


## 5. Random Forest Modeling

Three approaches are compared:

- Baseline Random Forest
- Class-balanced Random Forest for better churn recall
- Tuned Random Forest with deeper recall focus


In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    }
    return metrics, predictions, probabilities


models = {
    "Baseline RF": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Balanced RF": RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Tuned RF": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
}

results = []
fitted_models = {}
predictions_store = {}
probability_store = {}

for name, model in models.items():
    metrics, predictions, probabilities = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(metrics)
    fitted_models[name] = model
    predictions_store[name] = predictions
    probability_store[name] = probabilities

results_df = pd.DataFrame(results).sort_values(["Recall", "ROC-AUC"], ascending=False)
results_df


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_predictions = predictions_store[best_model_name]

print("Best model:", best_model_name)
print(classification_report(y_test, best_predictions, zero_division=0))
pd.DataFrame(
    confusion_matrix(y_test, best_predictions),
    index=["Actual Retained", "Actual Churned"],
    columns=["Predicted Retained", "Predicted Churned"],
)


## 6. Manual Hyperparameter Scan

The scan ranks models by recall first because missing churners is usually more expensive than contacting some retained customers.


In [ ]:
scan_rows = []

for n_estimators in [100, 200, 300, 400]:
    for max_depth in [5, 10, 15, None]:
        candidate = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=4,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )
        candidate.fit(X_train, y_train)
        pred = candidate.predict(X_test)
        prob = candidate.predict_proba(X_test)[:, 1]
        scan_rows.append(
            {
                "n_estimators": n_estimators,
                "max_depth": max_depth if max_depth is not None else "None",
                "Accuracy": accuracy_score(y_test, pred),
                "Recall": recall_score(y_test, pred, zero_division=0),
                "Precision": precision_score(y_test, pred, zero_division=0),
                "F1": f1_score(y_test, pred, zero_division=0),
                "ROC-AUC": roc_auc_score(y_test, prob),
            }
        )

scan_results = pd.DataFrame(scan_rows).sort_values(["Recall", "ROC-AUC", "Accuracy"], ascending=False)
scan_results.head(10)


## 7. Feature Importance and Smaller Model

Feature importance helps identify the most useful customer attributes and test whether a leaner model preserves performance.


In [ ]:
importance = (
    pd.DataFrame(
        {
            "Feature": X_encoded.columns,
            "Importance": best_model.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(importance.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=importance.head(15), x="Importance", y="Feature", color="#4C78A8")
plt.title("Top 15 Random Forest Features")
plt.tight_layout()
plt.show()


In [ ]:
low_importance_features = importance.tail(5)["Feature"].tolist()
print("Dropping low-importance features:", low_importance_features)

X_selected = X_encoded.drop(columns=low_importance_features)
X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X_selected,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

selected_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

selected_metrics, selected_predictions, selected_probabilities = evaluate_model(
    "Selected Feature RF",
    selected_rf,
    X_train_sel,
    X_test_sel,
    y_train_sel,
    y_test_sel,
)

pd.DataFrame([selected_metrics])


## 8. Cross-Validation

Five-fold cross-validation gives a more stable view of accuracy and recall across different train/test splits.


In [ ]:
cv_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

cv_summary = pd.DataFrame(
    {
        "Accuracy": cross_val_score(cv_model, X_encoded, y, cv=5, scoring="accuracy"),
        "Recall": cross_val_score(cv_model, X_encoded, y, cv=5, scoring="recall"),
        "ROC-AUC": cross_val_score(cv_model, X_encoded, y, cv=5, scoring="roc_auc"),
    }
)

display(cv_summary)
cv_summary.mean().to_frame("Mean Score")


## 9. ROC-AUC Curve

ROC-AUC measures how well the model separates churned and retained customers across probability thresholds.


In [ ]:
best_probabilities = probability_store[best_model_name]
fpr, tpr, thresholds = roc_curve(y_test, best_probabilities)
auc_score = roc_auc_score(y_test, best_probabilities)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"{best_model_name} AUC = {auc_score:.3f}", color="#1F77B4")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Customer Segmentation with K-Means

Churn probability is combined with tenure and billing values to create actionable customer segments.


In [ ]:
final_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
final_rf.fit(X_encoded, y)
churn_probability = final_rf.predict_proba(X_encoded)[:, 1]

segmentation_data = pd.DataFrame(
    {
        "Tenure Months": working_df["Tenure Months"],
        "Monthly Charges": working_df["Monthly Charges"],
        "Total Charges": working_df["Total Charges"],
        "Churn Probability": churn_probability,
    }
)

scaler = StandardScaler()
scaled_data = scaler.fit_transform(segmentation_data)

wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.tight_layout()
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
working_df["Cluster"] = kmeans.fit_predict(scaled_data)

cluster_profile = working_df.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Churn_Rate=("Churn Value", "mean"),
    Avg_Tenure=("Tenure Months", "mean"),
    Avg_Monthly_Charges=("Monthly Charges", "mean"),
    Avg_Total_Charges=("Total Charges", "mean"),
)

cluster_profile.sort_values("Churn_Rate", ascending=False).style.format(
    {
        "Churn_Rate": "{:.1%}",
        "Avg_Tenure": "{:.1f}",
        "Avg_Monthly_Charges": "${:,.2f}",
        "Avg_Total_Charges": "${:,.2f}",
    }
)


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=working_df.assign(Churn_Probability=churn_probability),
    x="Tenure Months",
    y="Churn_Probability",
    hue="Cluster",
    palette="Set2",
    alpha=0.75,
)
plt.title("Customer Segments: Tenure vs Churn Probability")
plt.ylabel("Churn Probability")
plt.tight_layout()
plt.show()


## Final Notes

For a churn model, recall and ROC-AUC matter more than accuracy alone. The best model should catch high-risk customers early enough for retention teams to act.
